In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Alternative text embeddings: GBB and network robustness

Calculate the SI S3.5.1–3 comparisons after notebook 04 has saved PaECTER and PatEmbed matches and CPC projections. Run from `revision_notebooks/` with paths set in the first cell.

Read the reference GBB and source-field assignments and source–GBB network from `BASELINE`, and alternative results from `DATA/embeddings/`. Matching IDs are DOCDB family IDs; `cos` is text cosine distance.


In [ ]:
import pandas as pd
from gbb_utils import (read_profile, ordinary_codes, nonnoise, extra_codes,
    code_profile, reduce_profile, cluster_profile, correspondence, correspondence_metrics,
    nmi_permutations, native_network, gbb_crosswalk, align_gbbs, network_metrics,
    membership_metrics)

MODELS = ["paecter", "patembed_base"]
PROJECTION_COLUMNS = ["ccmt_patent_id", "patent_id", "cpc", "leastsimilar", "cos"]
from gbb_utils import rca


## Match ranks and ordinary-CPC background memberships

In [ ]:
def add_submitted_ranks(projection, pairs):
    ranked = pairs.sort_values(["ccmt_patent_id", "cos", "patent_id"]).copy()
    ranked["rank"] = ranked.groupby("ccmt_patent_id").cumcount() + 1
    return projection.merge(ranked[["ccmt_patent_id", "patent_id", "rank"]], on=["ccmt_patent_id", "patent_id"])

def read_projection(path):
    # Archived projection notebooks omitted rank; obtain it from the retrieval table.
    projection = pd.read_parquet(path / "green_proj_cosangle.parquet", columns=PROJECTION_COLUMNS)
    pairs = pd.read_parquet(path / "matched_top10.parquet")
    return projection.merge(pairs[["ccmt_patent_id", "patent_id", "rank"]], on=["ccmt_patent_id", "patent_id"])

def background_codes(pairs, cpc):
    background = pairs[["ccmt_patent_id", "patent_id"]].merge(cpc.dropna(subset=["cpc"]), on="patent_id")
    background["cpc"] = ordinary_codes(background.cpc).str[:4]
    return background.loc[background.cpc.str.len().ge(4) & ~background.cpc.str.startswith("Y"),
                          ["ccmt_patent_id", "cpc"]].drop_duplicates()

def submitted_network():
    edges = pd.read_parquet(BASELINE / "srcpath33.parquet")
    edges = edges.rename(columns={"src": "source", "path": "gbb", "ccmt_patent_id": "edge_count"})
    return rca(edges, "source", "gbb", "edge_count")


## Construct and save the alternative GBB clusters

Use the projections from notebook 04 to construct GBBs. Save each model's `gbbs.parquet` beside its projection. The saved CPC layout defines the eligible code vocabulary.


In [ ]:
from gensim.models import Word2Vec

vectors = Word2Vec.load(str(BASELINE / "external/cpc_sg_ns_dim50.model")).wv
layout_codes = pd.read_parquet(BASELINE / "external/umap2d.parquet").cpc
for model in MODELS:
    path = DATA / "embeddings" / model
    projection = read_projection(path)
    profile = code_profile(extra_codes(projection))
    profile = profile.loc[profile.cpc.isin(layout_codes)].reset_index(drop=True)
    coordinates = reduce_profile(profile, vectors, seed=SEED)
    cluster_profile(profile, coordinates).to_parquet(path / "gbbs.parquet", index=False)


## Reference GBBs and extra-code abundance


In [ ]:
out = OUTPUTS / "embedding"
out.mkdir(parents=True, exist_ok=True)
original_pairs = pd.read_parquet(BASELINE / "matched_ccmt_other.parquet")
filtered = pd.read_parquet(BASELINE / "matched_ccmt_other_filtered.parquet")
projection = add_submitted_ranks(pd.read_parquet(BASELINE / "green_proj_cosangle.parquet", columns=PROJECTION_COLUMNS), original_pairs)
baseline_codes = extra_codes(projection)
reference = read_profile(BASELINE / "clusangle_outlier_hdbscan2.parquet")
sources = read_profile(BASELINE / "clusangle_outlier_hdbscan2_src2.parquet")
ref_network = submitted_network()
cpc = pd.read_parquet(PATSTAT / "tls225.parquet", columns=["docdb_family_id", "cpc_class_symbol"])
cpc.columns = ["patent_id", "cpc"]
background = background_codes(filtered, cpc)
baseline_top = original_pairs.sort_values(["ccmt_patent_id", "cos", "patent_id"]).groupby("ccmt_patent_id").head(1)
rows = []
abundance = [dict(model="fse", retained_focals=filtered.ccmt_patent_id.nunique(),
                  positive_focals=baseline_codes.ccmt_patent_id.nunique(), memberships=len(baseline_codes))]


## Compare each alternative

Extra-code weights are the maximum text similarity over selected matches, counting each family/code combination once. The network uses all focal-family CPC assignments in each projection. Match ranks come from the unfiltered retrieval tables, before applying the text-distance and shared-subclass filters.

In [ ]:
for i, model in enumerate(MODELS):
    path = DATA / "embeddings" / model
    pairs = pd.read_parquet(path / "matched_ccmt_other_filtered.parquet")
    raw_pairs = pd.read_parquet(path / "matched_top10.parquet")
    projected = read_projection(path)
    partition = read_profile(path / "gbbs.parquet")
    codes = extra_codes(projected)
    abundance.append(dict(model=model, retained_focals=pairs.ccmt_patent_id.nunique(),
                          positive_focals=codes.ccmt_patent_id.nunique(), memberships=len(codes)))
    top = baseline_top.merge(raw_pairs.loc[raw_pairs["rank"] == 1], on="ccmt_patent_id", suffixes=("_fse", "_new"))
    signal = membership_metrics(baseline_codes, codes)
    background_overlap = membership_metrics(background, background_codes(pairs, cpc))["mean_jaccard"]
    shared, cells = correspondence(reference, partition)
    shared = shared.sort_values("cpc")
    null_summary, null = nmi_permutations(shared, "clus_reference", "clus_alternative", "score", seed=20260914 + i)
    pd.DataFrame({"nmi": null}).to_csv(out / f"{model}_nmi_null.csv", index=False)
    cells.to_csv(out / f"{model}_correspondence.csv", index=False)
    focal = projected[["ccmt_patent_id", "cpc"]].drop_duplicates()
    native = native_network(focal, partition, sources)
    mapping = gbb_crosswalk(reference, partition)
    aligned = align_gbbs(native, mapping)
    native.to_parquet(out / f"{model}_native.parquet", index=False)
    mapping.to_parquet(out / f"{model}_crosswalk.parquet", index=False)
    rows.append(dict(model=model, same_top_match=top.patent_id_fse.eq(top.patent_id_new).mean(),
                     clusters=nonnoise(partition).clus.nunique(), extra_frequency_cosine=signal["frequency_cosine"],
                     extra_mean_jaccard=signal["mean_jaccard"], background_mean_jaccard=background_overlap,
                     **correspondence_metrics(cells), **null_summary,
                     **network_metrics(ref_network, aligned, sorted(nonnoise(reference).clus.unique()))))
totals = pd.DataFrame(abundance)
totals["memberships_per_retained_focal"] = totals.memberships / totals.retained_focals
totals.to_csv(out / "extra_code_abundance.csv", index=False)
pd.DataFrame(rows).to_csv(out / "embedding_comparison.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))
